# Multi-Label AMR Classifier — cross-site (A→B/C/D) and aggregated

**10-drug resistance prediction from MALDI-TOF spectra with a single shared
backbone and masked BCE.** Reference implementation for the documentation page
[docs/04-Multi-Label-mlp](https://asd-standard.github.io/Dryad_DRIAMS/04-Multi-Label-mlp/index.html).

## Protocol
- **Inputs** — `Processed/Proc_DRIAMS-{A..D}/{drug}/data.csv` (6000-bin
  spectra; NaN = untested drug). DRIAMS-A union: 9,443 spectra; pooled
  A+B+C+D union: 26,781.
- **Cross-site** — A split species-stratified 80/20 (train/val); B/C/D and
  their pooled union are test sets. Preprocessing, grid search and per-drug
  thresholds use A train/val only.
- **Aggregated** — A+B+C+D pooled, species-stratified 70/15/15; thresholds
  tuned on the aggregated validation split.
- **Models** — shared MLP backbone (6000→512→256→128, one logit per drug,
  masked BCE) vs a OneVsRest L2 logistic-regression baseline.
- **Metrics** — Balanced Accuracy with per-drug thresholds, plus AUC-ROC.

## Outputs (in `results_multilabel/`, not versioned in git)
`multilabel_results.csv`, `best_params_04.json`,
`Final-Comparison-Bal-Acc.txt`, `multilabel_grid_{cs,agg}.csv`,
`history_{crosssite,aggregated}.csv` and the heatmap/curve PDFs.

## Notes
- Grid search is 6×6 (lr × dropout) per setting; final runs retrain for up to
  100 epochs with warmup/cosine and keep the best-validation-loss checkpoint.
- Single seed (42) — reported metrics are point estimates; caveats are on the
  documentation page.
- The aggregated LR thresholds were corrected to validation-based tuning on
  2026-10-09; the standalone re-run lives in
  `04-00b-Aggregated-LR-ValTuned.ipynb`.

In [ ]:
!pip install maldideepkit maldiamrkit seaborn --quiet

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted')
    IN_COLAB = True
except ImportError:
    print('Running locally')
    IN_COLAB = False

In [ ]:
# Imports (numpy/pandas/torch/sklearn/MaldiDeepKit) and the global seed used
# by every stochastic step in this notebook.
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split, Subset
from tqdm import tqdm

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, accuracy_score)

from maldideepkit.attention.mlp import SpectralAttentionMLP
from maldideepkit.base.data import fit_input_transform, apply_input_transform
from maldiamrkit.evaluation import stratified_species_drug_split

# Single seed (42): all reported metrics are point estimates, not
# confidence intervals (see the caveats sections of the documentation).
warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

In [ ]:
# Paths, output directory and the canonical drug/site ordering used by the
# loader and every downstream array.
if IN_COLAB:
    DRYAD = Path("/content/drive/MyDrive/Flower/DRIAMS-DataSet")
    DATA_ROOT = DRYAD / "Processed"
else:
    DRYAD = Path("/media/asd/home/asd/Projects/Flower/DataSet/Dryad-DataSet")
    DATA_ROOT = DRYAD / "Processed"

# All artefacts (grids, histories, figures, summary CSVs) are written here.
# Results are intentionally not versioned in git (see repository README).
OUT_DIR = Path("./results_multilabel")
OUT_DIR.mkdir(exist_ok=True)

# Drug and site order define the label-matrix column order and the union order
# in load_multilabel_split() below - keep them stable for reproducibility.
BIN_COLS = [f"bin_{i}" for i in range(6000)]
DRUGS_10 = ["Ciprofloxacin", "Gentamicin", "Amoxicillin-Clavulanic acid",
            "Piperacillin-Tazobactam", "Cefepime", "Ceftriaxone",
            "Imipenem", "Ceftazidime", "Vancomycin", "Amikacin"]
SITES = ["DRIAMS-A", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D"]
print(f"Drugs: {len(DRUGS_10)}  |  Sites: {SITES}")
print(f"DATA_ROOT: {DATA_ROOT}")


In [ ]:
# =============================================================================
# 1. MULTI-LABEL DATASET (handles NaN = untested drugs)
# =============================================================================
# Defines MultiLabelDataset (spectra + labels + label mask) and
# load_multilabel_split(), which builds the per-site union matrices from the
# per-drug CSVs - deduplicating by ``code`` and storing NaN for untested drugs.

class MultiLabelDataset(Dataset):
    """Torch dataset yielding ``(spectrum, labels, label_mask)``.

    ``labels`` has NaNs replaced by 0 (those entries are never used by the
    loss); ``mask`` is 1.0 where the drug was tested and 0.0 otherwise, so
    untested drugs cannot contribute gradients.
    """

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.Y = torch.tensor(np.nan_to_num(Y, nan=0.0), dtype=torch.float32)
        self.mask = torch.tensor(~np.isnan(Y), dtype=torch.float32)
    def __len__(self):
        return len(self.X)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx], self.mask[idx]


def load_multilabel_split():
    """Build the multi-label matrices from the per-drug Processed CSVs.

    For each site, spectra from the ten per-drug files are deduplicated by
    ``code``: the first occurrence defines X and species, while a drug label
    seen again for the same code overwrites the previous value. The label
    matrix stores NaN for untested drugs.

    Returns
    -------
    train_X, train_Y, train_sp : ndarray
        DRIAMS-A union, shapes ``(n_A, 6000)``, ``(n_A, 10)`` (NaN where
        untested) and species strings.
    test_sites : dict
        ``{"DRIAMS-B": (X, Y, species), "DRIAMS-C": ..., "DRIAMS-D": ...}``.
    """
    train_X, train_Y, train_sp = None, None, None
    test_sites = {}

    drug_dirs = {d: d.replace(" ", "_") for d in DRUGS_10}  # spaces become underscores in directory names

    for site_name in SITES:
        proc_site = f"Proc_{site_name}"
        site_labels = {}   # {code: {drug: label}}
        X_samples = {}     # {code: X_vector}
        sp_samples = {}    # {code: species}
        site_drugs_ok = []

        for drug_name in DRUGS_10:
            p = DATA_ROOT / proc_site / drug_name.replace(" ", "_") / "data.csv"
            if not p.exists():
                continue
            df = pd.read_csv(p)
            site_drugs_ok.append(drug_name)
            for _, row in df.iterrows():
                code = row["code"]
                # First occurrence wins for X/species; duplicate rows for the
                # same code simply overwrite the drug label (last one wins).
                if code not in X_samples:
                    X_samples[code] = row[BIN_COLS].values.astype("float32")
                    sp_samples[code] = row["species"]
                site_labels.setdefault(code, {})[drug_name] = row["label"]

        if not site_drugs_ok:
            continue

        codes = list(X_samples.keys())
        X = np.array([X_samples[c] for c in codes], dtype="float32")
        species = np.array([sp_samples[c] for c in codes])

        # Build the label matrix (N, 10); NaN marks drugs not tested for that
        # spectrum and is excluded from the loss and from every metric.
        Y = np.full((len(codes), len(DRUGS_10)), np.nan, dtype=float)
        for i, drug in enumerate(DRUGS_10):
            for j, code in enumerate(codes):
                if drug in site_labels.get(code, {}):
                    Y[j, i] = site_labels[code][drug]

        n_lab = (~np.isnan(Y)).sum(axis=1).mean()
        print(f"  {site_name}: {len(codes)} samples, {len(site_drugs_ok)} drugs, avg {n_lab:.1f} labels/sample")

        if site_name == "DRIAMS-A":
            train_X, train_Y, train_sp = X, Y, species
        else:
            test_sites[site_name] = (X, Y, species)

    print(f"\nTrain (A): {train_X.shape[0]} samples, {len(np.unique(train_sp))} species")
    for s, (Xs, Ys, sps) in test_sites.items():
        print(f"  Test {s}: {Xs.shape[0]} samples, {len(np.unique(sps))} species")
    return train_X, train_Y, train_sp, test_sites

---
## Multi-Label MLP Class Definition

`MultiLabelDataset` and `MultiLabelMaldiMLP`: an sklearn-compatible wrapper
around MaldiDeepKit's MLP that trains with masked BCE and keeps the
best-validation-loss checkpoint.


In [ ]:
# =============================================================================
# 2. MULTI-LABEL MLP CLASSIFIER  (sklearn-compatible, PyTorch backend)
# =============================================================================
# Defines MultiLabelMaldiMLP: a sklearn-compatible wrapper around
# SpectralAttentionMLP with one logit per drug, masked-BCE training, a warmup +
# cosine schedule and best-validation-loss checkpointing.

class MultiLabelMaldiMLP(BaseEstimator, ClassifierMixin):
    """Sklearn-compatible multi-label MLP with a shared backbone.

    Wraps MaldiDeepKit's ``SpectralAttentionMLP`` (attention disabled here)
    with one output logit per drug and trains with masked binary
    cross-entropy: a sample contributes gradients only through the drug heads
    it was tested for.

    Parameters
    ----------
    hidden_dim : int, default=512
        Width of the shared projection layer.
    head_dims : tuple, default=(256, 128)
        Widths of the shared head layers (dropout_high after the first,
        dropout_low before the logits).
    use_attention : bool, default=False
        Enable sigmoid-gated attention on the projected features.
    dropout_high, dropout_low : float
        Dropout after the projection/first head layer and before the logits.
    learning_rate : float, default=1e-3
        Adam/AdamW learning rate.
    weight_decay : float, default=0.0
        If > 0, AdamW is used with this decoupled weight decay.
    batch_size : int, default=64
    epochs : int, default=50
        Maximum epochs; early stopping may stop sooner.
    early_stopping_patience : int, default=10
        Epochs without validation-loss improvement before stopping.
    warmup_epochs : int, default=0
        Linear LR warmup before cosine annealing.
    val_fraction : float, default=0.1
        Internal validation fraction for early stopping; when ``species`` is
        passed to :meth:`fit`, that split is species-stratified.
    random_state : int, default=42
    verbose : bool, default=False
    device : str, default="cpu"

    Attributes (set by :meth:`fit`)
    -------------------------------
    model_ : SpectralAttentionMLP
        Trained network with the best-validation-loss checkpoint restored.
    n_labels_ : int
        Number of output heads (10).
    history_ : dict
        Per-epoch ``train_loss``, ``val_loss`` and (when
        ``record_history=True``) ``val_macro_balacc`` at 0.5.
    best_epoch_, best_val_loss_, epochs_trained_
        Checkpoint bookkeeping (see the 04 loss-curves documentation page).
    is_fitted_ : bool

    Notes
    -----
    Predictions are per-drug probabilities via ``sigmoid``;
    :meth:`predict` applies either the default 0.5 cut or supplied per-drug
    thresholds. All test metrics in this notebook use thresholds tuned on the
    external validation split.
    """

    def __init__(self, hidden_dim=512, head_dims=(256, 128), use_attention=False,
                 dropout_high=0.3, dropout_low=0.2,
                 learning_rate=1e-3, weight_decay=0.0,
                 batch_size=64, epochs=50, early_stopping_patience=10,
                 warmup_epochs=0, val_fraction=0.1,
                 random_state=42, verbose=False, device="cpu"):
        self.hidden_dim = hidden_dim
        self.head_dims = head_dims
        self.use_attention = use_attention
        self.dropout_high = dropout_high
        self.dropout_low = dropout_low
        self.learning_rate = learning_rate
        self.weight_decay = weight_decay
        self.batch_size = batch_size
        self.epochs = epochs
        self.early_stopping_patience = early_stopping_patience
        self.warmup_epochs = warmup_epochs
        self.val_fraction = val_fraction
        self.random_state = random_state
        self.verbose = verbose
        self.device = device

    def _build_model(self, input_dim, n_classes):
        """Instantiate the backbone with one output logit per drug."""
        return SpectralAttentionMLP(
            input_dim=input_dim, n_classes=n_classes,
            hidden_dim=self.hidden_dim, head_dims=self.head_dims,
            use_attention=self.use_attention,
            dropout_high=self.dropout_high, dropout_low=self.dropout_low)

    @staticmethod
    def _macro_balacc(y_true, proba):
        """Macro Balanced Accuracy at 0.5 over drugs with usable labels.

        Masks NaN entries per drug and skips drugs with fewer than two samples
        or a single class on this split. Used for the recorded validation
        curve only; exported metrics use per-drug tuned thresholds.
        """
        bas = []
        for di in range(y_true.shape[1]):
            vm = ~np.isnan(y_true[:, di])
            if vm.sum() < 2:
                continue
            yv = y_true[vm, di].astype(int)
            if len(np.unique(yv)) < 2:
                continue
            bas.append(balanced_accuracy_score(yv, proba[vm, di] >= 0.5))
        return float(np.mean(bas)) if bas else np.nan

    def fit(self, X, Y, species=None, record_history=False):
        """Train with masked BCE, keeping the best-validation-loss checkpoint.

        Parameters
        ----------
        X : ndarray (N, 6000)
            Preprocessed spectra.
        Y : ndarray (N, 10)
            Resistance labels; NaN marks untested drugs.
        species : ndarray or None
            When given, the internal early-stopping split is
            species-stratified (mirrors the external protocol); otherwise a
            random split is used.
        record_history : bool, default=False
            Also record per-epoch validation macro Balanced Accuracy at 0.5
            (one extra evaluation pass; used for the final runs only).

        Returns
        -------
        self
        """
        np.random.seed(self.random_state)
        torch.manual_seed(self.random_state)

        Y_np = np.asarray(Y)
        self.n_labels_ = Y.shape[1]
        self.input_dim_ = X.shape[1]
        self.model_ = self._build_model(self.input_dim_, self.n_labels_).to(self.device)

        ds = MultiLabelDataset(X, Y)
        if species is not None:
            idx = np.arange(len(ds)).reshape(-1, 1)
            idx_tr, idx_val, _, _ = stratified_species_drug_split(
                idx, np.zeros(len(ds)), species=species,
                test_size=self.val_fraction, random_state=self.random_state)
            train_ds = Subset(ds, idx_tr.flatten().astype(int).tolist())
            val_ds = Subset(ds, idx_val.flatten().astype(int).tolist())
        else:
            n_val = max(1, int(len(ds) * self.val_fraction))
            n_tr = len(ds) - n_val
            train_ds, val_ds = random_split(ds, [n_tr, n_val],
                generator=torch.Generator().manual_seed(self.random_state))

        train_loader = DataLoader(train_ds, batch_size=self.batch_size, shuffle=True)
        val_loader = DataLoader(val_ds, batch_size=self.batch_size * 2, shuffle=False)

        opt_cls = torch.optim.AdamW if self.weight_decay > 0 else torch.optim.Adam
        optimizer = opt_cls(self.model_.parameters(), lr=self.learning_rate,
                            weight_decay=self.weight_decay)
        warmup = max(0, self.warmup_epochs)
        t_max = max(1, self.epochs - warmup)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=t_max, eta_min=1e-6)
        criterion = nn.BCEWithLogitsLoss(reduction="none")

        best_val_loss = float("inf")
        best_state = None
        patience_counter = 0

        self.history_ = {"train_loss": [], "val_loss": [], "val_macro_balacc": []}
        self.best_epoch_ = 0
        self.best_val_loss_ = float("inf")
        self.epochs_trained_ = 0

        for epoch in range(self.epochs):
            self.model_.train()
            train_loss = 0.0
            for xb, yb, mb in train_loader:
                xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                if epoch < warmup:
                    for pg in optimizer.param_groups:
                        pg["lr"] = self.learning_rate * (epoch + 1) / warmup
                optimizer.zero_grad()
                logits = self.model_(xb)
                # Masked BCE: average over known labels only, so untested
                # drugs contribute no gradient for a given sample.
                loss = criterion(logits, yb)
                loss = (loss * mb).sum() / mb.sum()
                loss.backward()
                optimizer.step()
                train_loss += loss.item()

            if epoch >= warmup:
                scheduler.step()

            self.model_.eval()
            val_loss = 0.0
            val_proba = []
            with torch.no_grad():
                for xb, yb, mb in val_loader:
                    xb, yb, mb = xb.to(self.device), yb.to(self.device), mb.to(self.device)
                    logits = self.model_(xb)
                    loss = criterion(logits, yb)
                    loss = (loss * mb).sum() / mb.sum()
                    val_loss += loss.item()
                    if record_history:
                        val_proba.append(torch.sigmoid(logits).cpu().numpy())

            val_loss /= len(val_loader)
            train_loss /= len(train_loader)

            self.history_["train_loss"].append(train_loss)
            self.history_["val_loss"].append(val_loss)
            if record_history:
                proba_full = np.concatenate(val_proba, axis=0)
                val_labels = Y_np[np.asarray(val_ds.indices)]
                self.history_["val_macro_balacc"].append(self._macro_balacc(val_labels, proba_full))

            if self.verbose:
                print(f"  Epoch {epoch+1:3d}/{self.epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            # Early stopping tracks validation loss (not Balanced Accuracy);
            # the best checkpoint is restored at the end of fit().
            if val_loss < best_val_loss:
                best_val_loss = val_loss
                best_state = {k: v.cpu().clone() for k, v in self.model_.state_dict().items()}
                self.best_epoch_ = epoch + 1
                self.best_val_loss_ = best_val_loss
                patience_counter = 0
            else:
                patience_counter += 1
                if patience_counter >= self.early_stopping_patience:
                    if self.verbose:
                        print(f"  Early stopping at epoch {epoch+1}")
                    break

        self.epochs_trained_ = len(self.history_["train_loss"])
        if best_state is not None:
            self.model_.load_state_dict(best_state)
        self.model_.eval()
        self.is_fitted_ = True
        return self

    def predict_proba(self, X):
        """Per-drug resistance probabilities, shape ``(N, 10)``."""
        self.model_.eval()
        X_t = torch.tensor(X, dtype=torch.float32).to(self.device)
        with torch.no_grad():
            logits = self.model_(X_t)
            proba = torch.sigmoid(logits).cpu().numpy()
        return proba

    def predict(self, X, thresholds=None):
        """0/1 predictions.

        ``thresholds`` may be a length-10 array of per-drug cut-offs (tuned on
        validation); by default every drug uses the 0.5 threshold.
        """
        proba = self.predict_proba(X)
        if thresholds is None:
            return (proba >= 0.5).astype(int)
        thresholds = np.asarray(thresholds)
        return (proba >= thresholds[None, :]).astype(int)


---
## Data Loading + Cross-Site Split

Train on A (species-stratified 80/20 train/val), test on B/C/D. Preprocessing
is fitted on A train only; the grid search and per-drug thresholds use A val.


In [ ]:
# =============================================================================
# 3. LOAD DATA + CROSS-SITE SPLIT (train A, test B/C/D)
# =============================================================================
# Loads the per-drug CSVs, builds the A union, splits it species-stratified
# 80/20 into train/val, fits log1p+standardization on the training split only,
# applies it to every split and adds the pooled B+C+D test union.

print("Loading data from Processed CSVs...")
X_train_raw, Y_train_raw, sp_train_raw, test_data_raw = load_multilabel_split()

# Species-stratified A split: train (80%) / val (20%). The validation split
# drives the grid search and the per-drug thresholds, then stays frozen; no
# species straddles the split boundary.
idx_tr, idx_v, _, _ = stratified_species_drug_split(
    np.arange(len(Y_train_raw)).reshape(-1, 1),
    np.zeros(len(Y_train_raw)),  # dummy labels — stratification is by species only
    species=sp_train_raw,
    test_size=0.20,
    random_state=SEED)
idx_tr = idx_tr.flatten().astype(int)
idx_v = idx_v.flatten().astype(int)

X_train = X_train_raw[idx_tr]
Y_train = Y_train_raw[idx_tr]
sp_train = sp_train_raw[idx_tr]
X_val = X_train_raw[idx_v]
Y_val = Y_train_raw[idx_v]
print(f"\nSplit: A train={len(X_train)}  A val={len(X_val)}")

# Preprocessing is fitted on the A training split only and applied unchanged
# to validation and test sites (no leakage across the split boundary).
state = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state)
X_val_pp = apply_input_transform(X_val, state)
print("Preprocessing done.")

# Apply preprocessing to test sites
test_pp = {}
for site, (Xs, Ys, _) in test_data_raw.items():
    test_pp[site] = (apply_input_transform(Xs, state), Ys)
    n_lab = (~np.isnan(Ys)).sum(axis=1).mean()
    print(f"  Test {site}: {Xs.shape[0]} samples preprocessed, avg {n_lab:.1f} labels/sample")

# B+C+D is a pooled convenience row weighted by site size (not a
# macro-average across sites).
if test_pp:
    X_combined = np.concatenate([t[0] for t in test_pp.values()])
    Y_combined = np.concatenate([t[1] for t in test_pp.values()])
    test_pp["B+C+D"] = (X_combined, Y_combined)
    print(f"  Test B+C+D combined: {X_combined.shape[0]} samples")


---
## Baseline: OneVsRest Logistic Regression

Linear reference model — ten independent L2 logistic regressions with the same
splits and threshold protocol as the MLP.


In [ ]:
# =============================================================================
# 4. BASELINE: OneVsRest Logistic Regression
# =============================================================================

# Trains one L2 logistic regression per drug (3-fold CV over C on the training
# split), tunes a per-drug threshold on the external A validation split and
# scores B/C/D with the frozen thresholds; results feed the summary table.
#
# Method details: docs/01.
C_grid = np.linspace(5e-5, 1e-3, 15)
thresholds = np.linspace(0.05, 0.95, 91)

lr_models = {}
lr_results = {}
for di, drug in enumerate(tqdm(DRUGS_10, desc="LR per drug")):
    # Training samples labeled for this drug
    tr_mask = ~np.isnan(Y_train[:, di])
    X_tr, y_tr = X_train_pp[tr_mask], Y_train[tr_mask, di].astype(int)
    val_mask = ~np.isnan(Y_val[:, di])
    X_v, y_v = X_val_pp[val_mask], Y_val[val_mask, di].astype(int)

    if len(np.unique(y_tr)) < 2 or len(np.unique(y_v)) < 2:
        lr_results[drug] = {"A-val_BalAcc": np.nan, "A-val_AUC": np.nan}
        continue

    # GridSearchCV
    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    lr = grid.best_estimator_

    # Threshold tuning
    proba_v = lr.predict_proba(X_v)[:, 1]
    best_t = thresholds[np.argmax([balanced_accuracy_score(y_v, proba_v >= t) for t in thresholds])]

    lr_models[drug] = (lr, best_t)
    lr_results[drug] = {"A-val_BalAcc": balanced_accuracy_score(y_v, proba_v >= best_t),
                        "A-val_AUC": roc_auc_score(y_v, proba_v)}

    for st, (X_st, Y_st) in test_pp.items():
        ts_mask = ~np.isnan(Y_st[:, di])
        if ts_mask.sum() < 2: continue
        X_ts, y_ts = X_st[ts_mask], Y_st[ts_mask, di].astype(int)
        proba = lr.predict_proba(X_ts)[:, 1]
        preds = proba >= best_t
        lr_results[drug][f"{st}_BalAcc"] = balanced_accuracy_score(y_ts, preds)
        lr_results[drug][f"{st}_AUC"] = roc_auc_score(y_ts, proba)

print("\nLR baseline done.")

In [ ]:
# ── LR Heatmap ──
# Renders and saves the per-drug LR Balanced Accuracy heatmap (A-val + B/C/D).
sites_order = ["A-val_BalAcc", "DRIAMS-B_BalAcc", "DRIAMS-C_BalAcc", "DRIAMS-D_BalAcc", "B+C+D_BalAcc"]
data = {}
for drug in DRUGS_10:
    row = [lr_results[drug].get(k, np.nan) for k in sites_order]
    data[drug] = row

fig, ax = plt.subplots(figsize=(10, 8))
hm = pd.DataFrame(data, index=[s.replace("_BalAcc", "") for s in sites_order]).T
sns.heatmap(hm, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
            linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title("LR OneVsRest -- Cross-Site (Train A, Test B/C/D)")
ax.set_xlabel("Test Set"); ax.set_ylabel("Drug")
plt.tight_layout()
plt.savefig(OUT_DIR / "lr_multilabel_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nLR Per-Drug BalAcc:")
print(hm.to_string())

---
## Shared MLP: Grid Search + Training

6×6 grid over learning rate × dropout, scored by macro Balanced Accuracy on
the external A validation split.


In [ ]:
# =============================================================================
# 5. SHARED MLP -- GRID SEARCH (lr x dropout)
# =============================================================================
# Runs the 6x6 lr x dropout grid on the cross-site split: each candidate is
# trained on A train and scored by per-drug thresholded macro Balanced Accuracy
# on A val; the full grid is saved for the documentation.

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Grid ranges follow the same pattern as the per-drug MLP studies (02).
# Weight decay 1e-3 during the search, 1e-4 for the final retrain below.
LR_GRID = np.linspace(1e-4, 5e-4, 6)
DROP_GRID = np.linspace(0.2, 0.6, 6)
print(f"Grid: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")

best_balacc = -1
best_lr = None; best_dh = None
grid_rows_cs = []

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        # Score with per-drug thresholds tuned on the external validation
        # split: the class imbalance makes one global threshold misleading.
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))

        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        grid_rows_cs.append({"lr": lr, "dropout_high": dh, "dropout_low": dl, "macro_BalAcc": macro_ba})
        mark = " *" if macro_ba > best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > best_balacc:
            best_balacc = macro_ba; best_lr = lr; best_dh = dh

print(f"\nBest: lr={best_lr:.1e} drop=({best_dh:.2f})  macro_BalAcc={best_balacc:.4f}")
pd.DataFrame(grid_rows_cs).to_csv(OUT_DIR / "multilabel_grid_cs.csv", index=False)


In [ ]:
# =============================================================================
# 6. RETRAIN BEST MLP + THRESHOLD TUNING + LOSS HISTORY
# =============================================================================

# Retrains the winning configuration (100 epochs, patience 15, wd 1e-4),
# records the per-epoch history, tunes per-drug thresholds on A val, evaluates
# every split and stores the checkpoint metadata. The exported model is the
# best-validation-loss checkpoint, not the epoch with the highest Balanced
# Accuracy (see the 04 loss-curves page).
mlp_best = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=best_dh, dropout_low=best_dh/2,
    learning_rate=best_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_best.fit(X_train_pp, Y_train, species=sp_train, record_history=True)

# Per-drug thresholds are tuned on the external A validation split and then
# frozen for every test split, so the reported test metrics are honest.
proba_val = mlp_best.predict_proba(X_val_pp)
mlp_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2:
        mlp_thresholds.append(0.5)
        continue
    pv = proba_val[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    mlp_thresholds.append(bt)
mlp_thresholds = np.array(mlp_thresholds)
print(f"\nPer-drug thresholds: {[f'{t:.2f}' for t in mlp_thresholds]}")

# Evaluate on all test sets (BalAcc + AUC)
mlp_results = {}
mlp_auc = {}
for st, (X_st, Y_st) in [("A-val", (X_val_pp, Y_val))] + list(test_pp.items()):
    proba = mlp_best.predict_proba(X_st)
    mlp_results[st] = {}
    mlp_auc[st] = {}
    for di, drug in enumerate(DRUGS_10):
        tm = ~np.isnan(Y_st[:, di])
        if tm.sum() < 2:
            mlp_results[st][drug] = np.nan; mlp_auc[st][drug] = np.nan
            continue
        preds = (proba[tm, di] >= mlp_thresholds[di])
        yt = Y_st[tm, di].astype(int)
        mlp_results[st][drug] = balanced_accuracy_score(yt, preds)
        mlp_auc[st][drug] = roc_auc_score(yt, proba[tm, di]) if len(np.unique(yt)) > 1 else np.nan

# ── Per-epoch histories: loss + validation macro Balanced Accuracy ──
hist = mlp_best.history_
ep = np.arange(1, len(hist["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(ep, hist["train_loss"], label="train", lw=2)
ax.plot(ep, hist["val_loss"], label="val", lw=2)
ax.axvline(mlp_best.best_epoch_, color="gray", ls=":", lw=1)
ax.annotate(f"best epoch {mlp_best.best_epoch_}", (mlp_best.best_epoch_, min(hist["val_loss"])),
            textcoords="offset points", xytext=(4, 6), fontsize=9, color="gray")
ax.set_title("Cross-site shared MLP — masked BCE loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, ls="--", alpha=0.4)
ax = axes[1]
ax.plot(ep, hist["val_macro_balacc"], color="#2ca02c", lw=2)
ax.axvline(mlp_best.best_epoch_, color="gray", ls=":", lw=1)
ax.set_title("Validation macro Balanced Accuracy @0.5")
ax.set_xlabel("Epoch"); ax.set_ylabel("macro BalAcc")
ax.grid(True, ls="--", alpha=0.4)
fig.suptitle("04 — cross-site shared MLP training curves", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_loss_curves_crosssite.pdf", bbox_inches="tight")
plt.show()
pd.DataFrame(hist).to_csv(OUT_DIR / "history_crosssite.csv", index=False)

# ``best_macro_balacc`` stored here is the maximum of the recorded validation
# curve; the exported test metrics come from the best-validation-loss
# checkpoint instead.
cs_best = {
    "lr": float(best_lr), "dropout": float(best_dh), "grid_macro_balacc": float(best_balacc),
    "best_epoch": int(mlp_best.best_epoch_), "best_val_loss": float(mlp_best.best_val_loss_),
    "epochs_trained": int(mlp_best.epochs_trained_),
    "best_macro_balacc": float(np.nanmax(hist["val_macro_balacc"])),
}
print("\nMLP done.")


In [ ]:
# ── MLP Heatmap ──
# Same heatmap for the shared MLP's per-drug Balanced Accuracy.
mlp_data = {}
for drug in DRUGS_10:
    row = [mlp_results[st].get(drug, np.nan) for st in ["A-val", "DRIAMS-B", "DRIAMS-C", "DRIAMS-D", "B+C+D"]]
    mlp_data[drug] = row

fig, ax = plt.subplots(figsize=(10, 8))
hm_mlp = pd.DataFrame(mlp_data, index=["A-val", "B", "C", "D", "B+C+D"]).T
sns.heatmap(hm_mlp, annot=True, fmt=".3f", cmap="RdYlGn", vmin=0.5, vmax=0.9,
            linewidths=0.5, cbar_kws={"label": "Balanced Accuracy"}, ax=ax)
ax.set_title("Shared MLP -- Cross-Site (Train A, Test B/C/D)")
ax.set_xlabel("Test Set"); ax.set_ylabel("Drug")
plt.tight_layout()
plt.savefig(OUT_DIR / "mlp_multilabel_heatmap.pdf", bbox_inches="tight")
plt.show()

print("\nMLP Per-Drug BalAcc:")
print(hm_mlp.to_string())

---
## LR vs MLP Comparison

In [ ]:
# =============================================================================
# 7. LR vs MLP COMPARISON
# =============================================================================

# Summary comparison for the documentation table. A-val columns are the split
# the thresholds were tuned on, so they are optimistic relative to B+C+D.
rows = []
for drug in DRUGS_10:
    lr_ba = lr_results[drug].get("B+C+D_BalAcc", np.nan)
    mlp_ba = mlp_results["B+C+D"].get(drug, np.nan)
    lr_av = lr_results[drug].get("A-val_BalAcc", np.nan)
    mlp_av = mlp_results["A-val"].get(drug, np.nan)
    rows.append({"Drug": drug, "LR_BalAcc": lr_ba, "MLP_BalAcc": mlp_ba,
                 "LR_Aval": lr_av, "MLP_Aval": mlp_av})

df_comp = pd.DataFrame(rows)
df_comp_display = df_comp.copy()
df_comp_display["Drug"] = df_comp_display["Drug"].str.slice(0, 15)
print(df_comp_display.to_string(index=False))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp)); w = 0.2
ax.bar(x - w, df_comp["LR_BalAcc"], w, label="LR (OneVsRest)", color="#1f77b4")
ax.bar(x,      df_comp["MLP_BalAcc"], w, label="MLP (Shared)", color="#ff7f0e")
ax.bar(x + w,  df_comp["LR_Aval"], w, label="LR A-val", color="#aec7e8", alpha=0.7)
ax.bar(x + 2*w, df_comp["MLP_Aval"], w, label="MLP A-val", color="#ffbb78", alpha=0.7)
ax.set_xticks(x)
ax.set_xticklabels([d[:15] for d in df_comp["Drug"]], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Balanced Accuracy (B+C+D)"); ax.set_title("LR vs Shared MLP -- Cross-Site Multi-Label")
ax.legend(fontsize=8); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_lr_vs_mlp.pdf")
plt.show()

In [ ]:
# Wrap-up: print the output directory and list the PDF figures that were
# written by the cross-site part of the notebook.
print("\nDone. Multi-label results saved to", OUT_DIR.resolve())
for f in sorted(OUT_DIR.glob("*.pdf")):
    print(f"  {f.name}")

---
## Aggregated Shared MLP (A+B+C+D → species-stratified 70/15/15)

The same model trained on the pooled four sites. Cross-site vs aggregated
isolates the domain-shift gap documented in 03/04.


In [ ]:
# =============================================================================
# 8. AGGREGATED SHARED MLP (all 4 sites A+B+C+D -> species-stratified 70/15/15)
# =============================================================================
# Re-runs the full pipeline on the pooled four sites: load union, species-
# stratified 70/15/15 split, grid search and final retrain with per-drug
# thresholds on the aggregated validation split, plus the cross-site-vs-
# aggregated comparison figure.

print("Loading data from all 4 sites...")
X_A, Y_A, sp_A, test_data_raw = load_multilabel_split()

# Aggregated protocol: pool A+B+C+D (26,781 spectra) and split
# species-stratified 70/15/15, so no species crosses a partition boundary.
X_all = np.concatenate([X_A] + [v[0] for v in test_data_raw.values()])
Y_all = np.concatenate([Y_A] + [v[1] for v in test_data_raw.values()])
sp_all = np.concatenate([sp_A] + [v[2] for v in test_data_raw.values()])
print(f"Total aggregated: {X_all.shape[0]} samples, {len(np.unique(sp_all))} species")

# Species-stratified split: train 70% / val 15% / test 15%
idx_trval, idx_test, _, _ = stratified_species_drug_split(
    np.arange(len(Y_all)).reshape(-1, 1),
    np.zeros(len(Y_all)),  # dummy labels
    species=sp_all,
    test_size=0.15,
    random_state=SEED)
idx_trval = idx_trval.flatten().astype(int)
idx_test = idx_test.flatten().astype(int)

X_trval, Y_trval, sp_trval = X_all[idx_trval], Y_all[idx_trval], sp_all[idx_trval]
X_test, Y_test = X_all[idx_test], Y_all[idx_test]

val_frac = 0.15 / 0.85
idx_train, idx_val, _, _ = stratified_species_drug_split(
    np.arange(len(Y_trval)).reshape(-1, 1),
    np.zeros(len(Y_trval)),
    species=sp_trval,
    test_size=val_frac,
    random_state=SEED)
idx_train = idx_train.flatten().astype(int); idx_val = idx_val.flatten().astype(int)
X_train, Y_train = X_trval[idx_train], Y_trval[idx_train]
sp_train_agg = sp_trval[idx_train]
X_val, Y_val = X_trval[idx_val], Y_trval[idx_val]

print(f"Split: train={X_train.shape[0]}  val={X_val.shape[0]}  test={X_test.shape[0]}")

# Transform fitted on the aggregated training split only.
state_agg = fit_input_transform(X_train, "log1p+standardize")
X_train_pp = apply_input_transform(X_train, state_agg)
X_val_pp   = apply_input_transform(X_val, state_agg)
X_test_pp  = apply_input_transform(X_test, state_agg)
print("Preprocessing done.")

# ── Grid search (6x6 lr x dropout, 36 combos) ──
print(f"\nGrid search: {len(LR_GRID)} lr x {len(DROP_GRID)} dropout = {len(LR_GRID)*len(DROP_GRID)} combos")
agg_best_balacc = -1; agg_best_lr = None; agg_best_dh = None
grid_rows_agg = []

for lr in LR_GRID:
    for d in DROP_GRID:
        dh, dl = d, d / 2
        mlp = MultiLabelMaldiMLP(
            hidden_dim=512, head_dims=(256, 128), use_attention=False,
            dropout_high=dh, dropout_low=dl,
            learning_rate=lr, weight_decay=1e-3,
            batch_size=64, epochs=50, early_stopping_patience=10,
            warmup_epochs=10, val_fraction=0.1,
            random_state=SEED, verbose=False, device=device)
        mlp.fit(X_train_pp, Y_train)

        proba_v = mlp.predict_proba(X_val_pp)
        per_drug_ba = []
        for di in range(10):
            vm = ~np.isnan(Y_val[:, di])
            if vm.sum() < 2: continue
            pv = proba_v[vm, di]; yv = Y_val[vm, di].astype(int)
            bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
            per_drug_ba.append(balanced_accuracy_score(yv, pv >= bt))
        macro_ba = np.mean(per_drug_ba) if per_drug_ba else 0.0
        grid_rows_agg.append({"lr": lr, "dropout_high": dh, "dropout_low": dl, "macro_BalAcc": macro_ba})
        mark = " *" if macro_ba > agg_best_balacc else ""
        print(f"  lr={lr:.1e}  drop=({dh:.2f},{dl:.2f})  macro_BalAcc={macro_ba:.4f}{mark}")
        if macro_ba > agg_best_balacc:
            agg_best_balacc = macro_ba; agg_best_lr = lr; agg_best_dh = dh

print(f"\nBest: lr={agg_best_lr:.1e} drop={agg_best_dh:.2f}  macro_BalAcc={agg_best_balacc:.4f}")
pd.DataFrame(grid_rows_agg).to_csv(OUT_DIR / "multilabel_grid_agg.csv", index=False)

# ── Retrain best MLP + threshold tuning ──
# Same retrain/checkpoint convention as the cross-site run; the per-drug
# thresholds below are tuned on the aggregated validation split.
mlp_agg = MultiLabelMaldiMLP(
    hidden_dim=512, head_dims=(256, 128), use_attention=False,
    dropout_high=agg_best_dh, dropout_low=agg_best_dh/2,
    learning_rate=agg_best_lr, weight_decay=1e-4,
    batch_size=64, epochs=100, early_stopping_patience=15,
    warmup_epochs=10, val_fraction=0.1,
    random_state=SEED, verbose=True, device=device)
mlp_agg.fit(X_train_pp, Y_train, species=sp_train_agg, record_history=True)

# Per-drug thresholds
proba_val_agg = mlp_agg.predict_proba(X_val_pp)
agg_thresholds = []
for di in range(10):
    vm = ~np.isnan(Y_val[:, di])
    if vm.sum() < 2:
        agg_thresholds.append(0.5); continue
    pv = proba_val_agg[vm, di]; yv = Y_val[vm, di].astype(int)
    bt = thresholds[np.argmax([balanced_accuracy_score(yv, pv >= t) for t in thresholds])]
    agg_thresholds.append(bt)
agg_thresholds = np.array(agg_thresholds)

# Evaluate on test set (BalAcc + AUC)
proba_test = mlp_agg.predict_proba(X_test_pp)
agg_results = {}
agg_auc = {}
for di, drug in enumerate(DRUGS_10):
    tm = ~np.isnan(Y_test[:, di])
    if tm.sum() < 2:
        agg_results[drug] = np.nan; agg_auc[drug] = np.nan; continue
    preds = (proba_test[tm, di] >= agg_thresholds[di])
    yt = Y_test[tm, di].astype(int)
    agg_results[drug] = balanced_accuracy_score(yt, preds)
    agg_auc[drug] = roc_auc_score(yt, proba_test[tm, di]) if len(np.unique(yt)) > 1 else np.nan

print("\nAggregated Shared MLP -- Test BalAcc:")
for drug, ba in agg_results.items():
    print(f"  {drug:35s}  {ba:.4f}")

# ── Per-epoch histories ──
hist_agg = mlp_agg.history_
ep = np.arange(1, len(hist_agg["train_loss"]) + 1)
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ax = axes[0]
ax.plot(ep, hist_agg["train_loss"], label="train", lw=2)
ax.plot(ep, hist_agg["val_loss"], label="val", lw=2)
ax.axvline(mlp_agg.best_epoch_, color="gray", ls=":", lw=1)
ax.annotate(f"best epoch {mlp_agg.best_epoch_}", (mlp_agg.best_epoch_, min(hist_agg["val_loss"])),
            textcoords="offset points", xytext=(4, 6), fontsize=9, color="gray")
ax.set_title("Aggregated shared MLP — masked BCE loss")
ax.set_xlabel("Epoch"); ax.set_ylabel("Loss")
ax.legend(); ax.grid(True, ls="--", alpha=0.4)
ax = axes[1]
ax.plot(ep, hist_agg["val_macro_balacc"], color="#2ca02c", lw=2)
ax.axvline(mlp_agg.best_epoch_, color="gray", ls=":", lw=1)
ax.set_title("Validation macro Balanced Accuracy @0.5")
ax.set_xlabel("Epoch"); ax.set_ylabel("macro BalAcc")
ax.grid(True, ls="--", alpha=0.4)
fig.suptitle("04 — aggregated shared MLP training curves", fontsize=13, fontweight="bold")
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_loss_curves_aggregated.pdf", bbox_inches="tight")
plt.show()
pd.DataFrame(hist_agg).to_csv(OUT_DIR / "history_aggregated.csv", index=False)

agg_best = {
    "lr": float(agg_best_lr), "dropout": float(agg_best_dh), "grid_macro_balacc": float(agg_best_balacc),
    "best_epoch": int(mlp_agg.best_epoch_), "best_val_loss": float(mlp_agg.best_val_loss_),
    "epochs_trained": int(mlp_agg.epochs_trained_),
    "best_macro_balacc": float(np.nanmax(hist_agg["val_macro_balacc"])),
}

# ── Aggregated vs Cross-Site comparison ──
comp_rows = []
for drug in DRUGS_10:
    cs_ba = mlp_results["B+C+D"].get(drug, np.nan)
    ag_ba = agg_results.get(drug, np.nan)
    comp_rows.append({"Drug": drug, "Cross-Site (A->B+C+D)": cs_ba, "Aggregated (70/15/15)": ag_ba})
df_comp2 = pd.DataFrame(comp_rows)
df_comp2["Drug"] = df_comp2["Drug"].str.slice(0, 15)

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(df_comp2)); w = 0.35
ax.bar(x - w/2, df_comp2["Cross-Site (A->B+C+D)"], w, label="Cross-Site A->B+C+D", color="#1f77b4")
ax.bar(x + w/2, df_comp2["Aggregated (70/15/15)"], w, label="Aggregated 70/15/15", color="#ff7f0e")
ax.set_xticks(x); ax.set_xticklabels(df_comp2["Drug"], fontsize=8, rotation=45, ha="right")
ax.set_ylabel("Test BalAcc"); ax.set_title("Shared MLP -- Cross-Site vs Aggregated")
ax.legend(); ax.set_ylim(0, 1); ax.axhline(0.5, color="gray", ls="--", alpha=0.4)
plt.grid(True, ls='--', lw=0.5, color='gray', alpha=0.7)
plt.tight_layout(); plt.savefig(OUT_DIR / "multilabel_cross_vs_agg.pdf")
plt.show()

print("\nCross-Site vs Aggregated comparison:")
print(df_comp2.to_string(index=False))


---
## Aggregated OneVsRest LR + Persistence

Linear baseline on the aggregated split, followed by the export of every
artefact consumed by the documentation.


In [ ]:
# =============================================================================
# 9. AGGREGATED OneVsRest LOGISTIC REGRESSION
# =============================================================================
# Fits the OneVsRest LR baseline on the aggregated split and records per-drug
# BalAcc/AUC for the final summary table.

# Note (2026-10-09): thresholds are tuned on the aggregated validation split,
# never on the test set. The original export tuned on test and was corrected;
# see 04-00b-Aggregated-LR-ValTuned.ipynb and the docs provenance section.
agg_lr_results = {}
agg_lr_auc = {}
for di, drug in enumerate(tqdm(DRUGS_10, desc="AGG LR per drug")):
    tr_mask = ~np.isnan(Y_train[:, di])
    X_tr, y_tr = X_train_pp[tr_mask], Y_train[tr_mask, di].astype(int)
    val_mask = ~np.isnan(Y_val[:, di])
    X_v, y_v = X_val_pp[val_mask], Y_val[val_mask, di].astype(int)
    te_mask = ~np.isnan(Y_test[:, di])
    X_te, y_te = X_test_pp[te_mask], Y_test[te_mask, di].astype(int)

    if len(np.unique(y_tr)) < 2 or len(np.unique(y_te)) < 2:
        agg_lr_results[drug] = np.nan; agg_lr_auc[drug] = np.nan
        continue

    grid = GridSearchCV(
        LogisticRegression(penalty="l2", solver="lbfgs", class_weight="balanced", max_iter=5000, random_state=SEED),
        param_grid={"C": C_grid}, cv=3, scoring="balanced_accuracy", n_jobs=-1, verbose=0)
    grid.fit(X_tr, y_tr)
    lr = grid.best_estimator_

    # Threshold: validation split only, then frozen for the test set
    proba_v = lr.predict_proba(X_v)[:, 1]
    best_t = thresholds[np.argmax([balanced_accuracy_score(y_v, proba_v >= t) for t in thresholds])]

    proba = lr.predict_proba(X_te)[:, 1]
    agg_lr_results[drug] = balanced_accuracy_score(y_te, proba >= best_t)
    agg_lr_auc[drug] = roc_auc_score(y_te, proba)

print("\nAGG LR done.")


In [ ]:
# =============================================================================
# 10. PERSISTENCE: RESULTS, CONFIGS, FINAL COMPARISON
# =============================================================================
# ---------------------------------------------------------------------------
# Persist the artefacts used by the Sphinx pages in docs/04:
#   multilabel_results.csv        per-drug BalAcc/AUC for all combinations
#   best_params_04.json           winning grid configs (cross-site, aggregated)
#   Final-Comparison-Bal-Acc.txt  human-readable CS-vs-AGG summary
# AGG_LR_* uses validation-tuned thresholds (corrected 2026-10-09).
# ---------------------------------------------------------------------------
import json

rows = []
for drug in DRUGS_10:
    rows.append({
        "Drug": drug,
        "CS_LR_BalAcc": lr_results[drug].get("B+C+D_BalAcc", np.nan),
        "CS_LR_AUC": lr_results[drug].get("B+C+D_AUC", np.nan),
        "CS_MLP_BalAcc": mlp_results["B+C+D"].get(drug, np.nan),
        "CS_MLP_AUC": mlp_auc["B+C+D"].get(drug, np.nan),
        "AGG_LR_BalAcc": agg_lr_results.get(drug, np.nan),
        "AGG_LR_AUC": agg_lr_auc.get(drug, np.nan),
        "AGG_MLP_BalAcc": agg_results.get(drug, np.nan),
        "AGG_MLP_AUC": agg_auc.get(drug, np.nan),
    })
df_results = pd.DataFrame(rows)
df_results.to_csv(OUT_DIR / "multilabel_results.csv", index=False)
print(df_results.to_string(index=False))

with open(OUT_DIR / "best_params_04.json", "w") as f:
    json.dump({"cross_site": cs_best, "aggregated": agg_best}, f, indent=2)

out_path = OUT_DIR / "Final-Comparison-Bal-Acc.txt"
with open(out_path, "w") as f:
    f.write("Final Comparison:  Cross-Site (CS) vs Aggregated (AGG)\n\n")
    f.write("           Drug:      CS_LR   CS_MLP   AGG_LR  AGG_MLP\n")
    for _, r in df_results.iterrows():
        line = (f"{r['Drug']:>20s}  {r['CS_LR_BalAcc']:8.6f}  {r['CS_MLP_BalAcc']:8.6f}  "
                f"{r['AGG_LR_BalAcc']:8.6f}  {r['AGG_MLP_BalAcc']:8.6f}")
        print(line)
        f.write(line + "\n")

print("\nSaved: multilabel_results.csv, best_params_04.json, Final-Comparison-Bal-Acc.txt")
